In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('rds_data_clean.csv')
print(df.shape)
print(df.dtypes)


(1369, 31)
Sl.No                         int64
Ref No Data                     str
Entry code                    int64
IP No                       float64
Birth Term                      str
Gestational Age (Weeks)     float64
Category                        str
Place                           str
Gender                          str
Type                            str
Presentation                    str
Other Complications/Info        str
Birth weight (gms)          float64
Apgar at 1                  float64
Apgar at 5                  float64
Heart rate                      str
Respiratory rate                str
SpO2                            str
Head circumference          float64
Length                      float64
Mother Age                  float64
Mother Hb                   float64
Mother Blood Group              str
Parity                          str
Arterial Blood Ph           float64
Arterial Blood Pco2         float64
Arterial Blood Po2          float64
Target           

In [2]:
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'count': missing, 'pct': missing_pct})
print(missing_df[missing_df['count'] > 0].sort_values('pct', ascending=False))


                          count    pct
Other Complications/Info   1170  85.46
Length                     1105  80.72
Mother Hb                   866  63.26
Head circumference          509  37.18
SpO2_pct                    387  28.27
Gestational Age (Weeks)     346  25.27
Mother Age                  325  23.74
Arterial Blood Po2          321  23.45
Arterial Blood Pco2         317  23.16
Arterial Blood Ph           312  22.79
SpO2                        299  21.84
Apgar at 1                  218  15.92
Apgar at 5                  218  15.92
Mother Blood Group          216  15.78
Presentation                212  15.49
Place                       181  13.22
Parity                      146  10.66
Birth weight (gms)          118   8.62
Category                     89   6.50
Type                         87   6.36
IP No                        85   6.21
Resp_rate_cpm                70   5.11
Respiratory rate             60   4.38
Gender                       57   4.16
Heart_rate_bpm           

In [3]:
cat_cols = ['Birth Term', 'Category', 'Place', 'Gender', 'Type',
            'Presentation', 'Mother Blood Group', 'Parity', 'Target']

for col in cat_cols:
    vals = df[col].dropna().astype(str).str.strip().str.lower()
    vc = vals.value_counts()
    print(f'\n--- {col} | {len(vc)} unique | {df[col].isna().sum()} missing ---')
    print(vc.to_string())



--- Birth Term | 22 unique | 44 missing ---
Birth Term
preterm                433
term                   413
late preterm           288
early preterm          104
extreme preterm         61
late pre term            4
extream preterm          3
late term                3
full term                2
moderate preterm         2
remove(above age)        1
extreme prematurity      1
terms                    1
latepreterm              1
(no template)            1
post term                1
exterm preterm           1
extremely preterm        1
earlyterm                1
exteme preterm           1
septic shock             1
late pre-term            1

--- Category | 39 unique | 89 missing ---
Category
aga                                                     1002
sga                                                      191
lga                                                       26
ept-aga                                                    9
rds                                                   

In [4]:
num_cols = ['Gestational Age (Weeks)', 'Birth weight (gms)', 'Apgar at 1', 'Apgar at 5',
            'Head circumference', 'Mother Age', 'Mother Hb',
            'Arterial Blood Ph', 'Arterial Blood Pco2', 'Arterial Blood Po2',
            'Heart_rate_bpm', 'Resp_rate_cpm', 'SpO2_pct']

for col in num_cols:
    s = df[col].dropna()
    print(f'{col}: min={s.min():.2f}, max={s.max():.2f}, mean={s.mean():.2f}, n={len(s)}')


Gestational Age (Weeks): min=21.00, max=41.00, mean=33.00, n=1023
Birth weight (gms): min=300.00, max=5126.00, mean=2101.09, n=1251
Apgar at 1: min=0.00, max=11.00, mean=7.24, n=1151
Apgar at 5: min=1.00, max=20.00, mean=8.31, n=1151
Head circumference: min=23.00, max=143.00, mean=32.24, n=860
Mother Age: min=19.00, max=44.00, mean=28.31, n=1044
Mother Hb: min=4.90, max=1301.00, mean=18.84, n=503
Arterial Blood Ph: min=0.00, max=36.00, mean=7.23, n=1057
Arterial Blood Pco2: min=7.27, max=522.00, mean=49.61, n=1052
Arterial Blood Po2: min=7.00, max=292.00, mean=49.71, n=1048
Heart_rate_bpm: min=15.00, max=1425.00, mean=144.13, n=1317
Resp_rate_cpm: min=2.00, max=140.00, mean=63.91, n=1299
SpO2_pct: min=18.00, max=100.00, mean=88.70, n=982


In [5]:
df2 = df.copy()

# Drop: identifiers, raw text vitals (already extracted), too sparse, free text
drop_cols = ['Sl.No', 'Ref No Data', 'Entry code', 'IP No',
             'Heart rate', 'Respiratory rate', 'SpO2',
             'Other Complications/Info', 'Length']

df2 = df2.drop(columns=[c for c in drop_cols if c in df2.columns])
print(df2.shape)
print(df2.columns.tolist())


(1369, 22)
['Birth Term', 'Gestational Age (Weeks)', 'Category', 'Place', 'Gender', 'Type', 'Presentation', 'Birth weight (gms)', 'Apgar at 1', 'Apgar at 5', 'Head circumference', 'Mother Age', 'Mother Hb', 'Mother Blood Group', 'Parity', 'Arterial Blood Ph', 'Arterial Blood Pco2', 'Arterial Blood Po2', 'Target', 'Heart_rate_bpm', 'Resp_rate_cpm', 'SpO2_pct']


In [6]:
bounds = {
    'Apgar at 1':           (0, 10),
    'Apgar at 5':           (0, 10),
    'Head circumference':   (23, 40),
    'Mother Hb':            (4, 16),
    'Arterial Blood Ph':    (6.8, 7.8),
    'Arterial Blood Pco2':  (15, 80),
    'Arterial Blood Po2':   (20, 150),
    'Heart_rate_bpm':       (60, 250),
    'Resp_rate_cpm':        (20, 120),
    'SpO2_pct':             (50, 100),
}

for col, (lo, hi) in bounds.items():
    before = df2[col].notna().sum()
    df2.loc[(df2[col] < lo) | (df2[col] > hi), col] = np.nan
    after = df2[col].notna().sum()
    print(f'{col}: {before - after} values clipped to NaN')


Apgar at 1: 1 values clipped to NaN
Apgar at 5: 1 values clipped to NaN
Head circumference: 10 values clipped to NaN
Mother Hb: 283 values clipped to NaN
Arterial Blood Ph: 7 values clipped to NaN
Arterial Blood Pco2: 29 values clipped to NaN
Arterial Blood Po2: 36 values clipped to NaN
Heart_rate_bpm: 11 values clipped to NaN
Resp_rate_cpm: 6 values clipped to NaN
SpO2_pct: 24 values clipped to NaN


In [7]:
def clean_birth_term(val):
    if pd.isna(val):
        return np.nan
    v = str(val).lower().strip()
    v = v.replace('-', ' ').replace('_', ' ')
    # collapse whitespace
    import re
    v = re.sub(r'\s+', ' ', v)

    if any(x in v for x in ['extreme prematuri', 'extrem', 'exterm', 'exteme', 'evpt']):
        return 'extreme preterm'
    if 'early' in v or 'earlyterm' in v:
        return 'early preterm'
    if 'late pre' in v or 'latepreterm' in v or 'lpt' in v:
        return 'late preterm'
    if 'late term' in v:
        return 'late preterm'
    if 'moderate' in v:
        return 'late preterm'   # moderate preterm ~ late preterm clinically
    if 'preterm' in v or 'prterm' in v:
        return 'preterm'
    if 'full term' in v or 'term' == v or 'terms' in v:
        return 'term'
    if 'post term' in v:
        return 'term'
    return np.nan  # garbage values like 'septic shock', 'remove(above age)'

df2['Birth Term'] = df2['Birth Term'].apply(clean_birth_term)
print(df2['Birth Term'].value_counts(dropna=False))


Birth Term
preterm            436
term               417
late preterm       299
early preterm      105
extreme preterm     65
NaN                 47
Name: count, dtype: int64


In [8]:
def clean_category(val):
    if pd.isna(val):
        return np.nan
    v = str(val).upper().strip()
    if 'SGA' in v or 'IUGR' in v or 'IGR' in v:
        return 'SGA'
    if 'LGA' in v:
        return 'LGA'
    if 'AGA' in v:
        return 'AGA'
    return np.nan  # RDS, IDM, MCDA etc. — not weight-for-age categories

df2['Category'] = df2['Category'].apply(clean_category)
print(df2['Category'].value_counts(dropna=False))


Category
AGA    1027
SGA     204
NaN     112
LGA      26
Name: count, dtype: int64


In [9]:
def clean_type(val):
    if pd.isna(val):
        return np.nan
    v = str(val).lower().strip()
    if any(x in v for x in ['lscs', 'caesarean', 'cesarean', 'c-section', 'c-sec', 'lcls']):
        return 'lscs'
    if any(x in v for x in ['nvd', 'vaginal', 'svd', 'vavd', 'vacuum', 'vaccum', 'normal']):
        return 'nvd'
    return np.nan  # twin, single, singleton, pprom, male, aga — not delivery types

df2['Type'] = df2['Type'].apply(clean_type)
print(df2['Type'].value_counts(dropna=False))


Type
lscs    789
nvd     464
NaN     116
Name: count, dtype: int64


In [10]:
def clean_presentation(val):
    if pd.isna(val):
        return np.nan
    v = str(val).lower().strip()
    if 'vertex' in v or 'cephalic' in v:
        return 'vertex'
    if 'breech' in v:
        return 'breech'
    return 'other'

df2['Presentation'] = df2['Presentation'].apply(clean_presentation)
print(df2['Presentation'].value_counts(dropna=False))

# ── Mother Blood Group ──────────────────────────────────────────────
def clean_blood_group(val):
    if pd.isna(val):
        return np.nan
    v = str(val).lower().strip()
    v = v.replace("'", '').replace('rh(d)', '').replace('rh (d)', '').strip()
    v = v.replace('negetive', 'negative').replace('neagtive', 'negative')
    v = v.replace('negatve', 'negative').replace('negtive', 'negative')
    v = v.replace('postitive', 'positive').replace('positiv', 'positive')
    v = v.replace('+ve', 'positive').replace('-ve', 'negative')
    v = v.replace('opositive', 'o positive')
    v = v.replace('+', ' positive').replace('-', ' negative')
    v = v.replace('0 ', 'o ').replace('0p', 'o p')

    import re
    v = re.sub(r'\s+', ' ', v).strip()

    # extract ABO and Rh
    abo = None
    for g in ['ab', 'a1b', 'a1', 'a', 'b', 'o']:
        if v.startswith(g):
            abo = 'ab' if g in ('ab', 'a1b') else g[0]
            break
    if abo is None:
        return np.nan

    rh = 'positive' if 'positive' in v else ('negative' if 'negative' in v else None)
    if rh is None:
        return np.nan

    return f'{abo} {rh}'

df2['Mother Blood Group'] = df2['Mother Blood Group'].apply(clean_blood_group)
print(df2['Mother Blood Group'].value_counts(dropna=False))


Presentation
vertex    1093
NaN        212
breech      62
other        2
Name: count, dtype: int64
Mother Blood Group
o positive     426
b positive     300
a positive     268
NaN            224
ab positive     71
o negative      29
b negative      25
a negative      21
ab negative      5
Name: count, dtype: int64


In [11]:
import re

def clean_parity(val):
    if pd.isna(val):
        return np.nan
    v = str(val).lower().strip()

    # junk entries
    if any(x in v for x in ['gdm', 'oligohydr', 'preelamp', 'pprom', 'ph', 'zero', 'unknown', 'gravida ']):
        return np.nan

    if 'primi' in v or v == 'g1' or v == 'g1p0' or v == 'p1' or v == '1':
        return 1

    # extract leading gravida number from patterns like g2p1l1, g3p2l2, g2a1 etc.
    m = re.match(r'g[\-]?(\d+)', v)
    if m:
        return int(m.group(1))

    # bare numbers like '2', '3', '4'
    if re.fullmatch(r'\d', v):
        return int(v)

    # patterns like p2l2, p2 — no gravida info, skip
    return np.nan

df2['Parity'] = df2['Parity'].apply(clean_parity)
print(df2['Parity'].value_counts(dropna=False).sort_index())


Parity
1.0     567
2.0     326
3.0     176
4.0      85
5.0      18
6.0       9
7.0       2
8.0       1
43.0      1
NaN     184
Name: count, dtype: int64


In [12]:
from sklearn.impute import SimpleImputer

# ── Categorical: mode imputation ────────────────────────────────────
cat_cols = ['Birth Term', 'Category', 'Place', 'Gender', 'Type',
            'Presentation', 'Mother Blood Group']

for col in cat_cols:
    mode = df2[col].mode()[0]
    df2[col] = df2[col].fillna(mode)
    print(f'{col}: filled with mode = "{mode}"')

# ── Numeric: median imputation ──────────────────────────────────────
num_cols = ['Gestational Age (Weeks)', 'Birth weight (gms)', 'Apgar at 1', 'Apgar at 5',
            'Head circumference', 'Mother Age', 'Mother Hb',
            'Arterial Blood Ph', 'Arterial Blood Pco2', 'Arterial Blood Po2',
            'Heart_rate_bpm', 'Resp_rate_cpm', 'SpO2_pct', 'Parity']

for col in num_cols:
    median = df2[col].median()
    df2[col] = df2[col].fillna(median)
    print(f'{col}: filled with median = {median:.2f}')

print('\nRemaining nulls:')
print(df2.isna().sum()[df2.isna().sum() > 0])


Birth Term: filled with mode = "preterm"
Category: filled with mode = "AGA"
Place: filled with mode = "inborn"
Gender: filled with mode = "male"
Type: filled with mode = "lscs"
Presentation: filled with mode = "vertex"
Mother Blood Group: filled with mode = "o positive"
Gestational Age (Weeks): filled with median = 34.00
Birth weight (gms): filled with median = 2055.00
Apgar at 1: filled with median = 8.00
Apgar at 5: filled with median = 9.00
Head circumference: filled with median = 32.00
Mother Age: filled with median = 28.00
Mother Hb: filled with median = 14.40
Arterial Blood Ph: filled with median = 7.21
Arterial Blood Pco2: filled with median = 48.00
Arterial Blood Po2: filled with median = 42.00
Heart_rate_bpm: filled with median = 145.00
Resp_rate_cpm: filled with median = 65.00
SpO2_pct: filled with median = 94.00
Parity: filled with median = 2.00

Remaining nulls:
Series([], dtype: int64)


In [13]:
print('Shape:', df2.shape)
print('\nValue counts — Target:')
print(df2['Target'].value_counts())

print('\nSample:')
print(df2.head(3))

df2.to_csv('rds_data_clean2.csv', index=False)
print('\nSaved to rds_data_clean2.csv')


Shape: (1369, 22)

Value counts — Target:
Target
positive    1000
negative     369
Name: count, dtype: int64

Sample:
  Birth Term  Gestational Age (Weeks) Category   Place  Gender  Type  \
0    preterm                     29.0      SGA  inborn    male  lscs   
1    preterm                     32.0      SGA  inborn  female  lscs   
2    preterm                     32.0      AGA  inborn  female  lscs   

  Presentation  Birth weight (gms)  Apgar at 1  Apgar at 5  ...  Mother Hb  \
0       vertex               880.0         3.0         7.0  ...       14.4   
1       breech              1095.0         6.0         8.0  ...       14.4   
2       breech              1500.0         8.0         9.0  ...       14.4   

   Mother Blood Group  Parity Arterial Blood Ph  Arterial Blood Pco2  \
0          o positive     2.0              7.21                 35.0   
1          o positive     2.0              7.42                 25.0   
2          o positive     2.0              7.21                 

In [14]:
df3=pd.read_csv('rds_data_clean2.csv')
df3.head()
df3.isnull().sum()

Birth Term                 0
Gestational Age (Weeks)    0
Category                   0
Place                      0
Gender                     0
Type                       0
Presentation               0
Birth weight (gms)         0
Apgar at 1                 0
Apgar at 5                 0
Head circumference         0
Mother Age                 0
Mother Hb                  0
Mother Blood Group         0
Parity                     0
Arterial Blood Ph          0
Arterial Blood Pco2        0
Arterial Blood Po2         0
Target                     0
Heart_rate_bpm             0
Resp_rate_cpm              0
SpO2_pct                   0
dtype: int64